In [ ]:
# ==========================================
# MULE ACCOUNT DETECTION
# PHASE 4 : FEATURE SELECTION
# ==========================================

import pandas as pd
import numpy as np

from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------
# LOAD DATASET
# ------------------------------------------

df = pd.read_csv(
    "../data/processed/preprocessed_data.csv"
)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Shape:", df.shape)

# ------------------------------------------
# TARGET COLUMN
# ------------------------------------------

target_col = "F3924"

# ------------------------------------------
# DISCOVERED FEATURES
# ------------------------------------------

selected_features = [

'F1057','F1063','F1104','F116','F1165',
'F1177','F1212','F1320','F1381','F1387',
'F1393','F1428','F1489','F1495','F1501',
'F1536','F157','F158','F159','F1597',
'F160','F1603','F1605','F1609','F1611',
'F162','F1707','F1711','F1713','F1717',
'F1719','F1753','F1755','F1814','F1815',
'F1819','F1821','F1825','F1827','F1861',
'F1863','F1966','F1968','F2030','F2074',
'F2076','F2143','F2149','F2182','F2184',
'F2230','F224','F2285','F230','F2481',
'F2486','F2489','F2578','F265','F266',
'F267','F268','F2686','F270','F2779',
'F3226','F3228','F3229','F3231','F3337',
'F3339','F3445','F3447','F3484','F3490',
'F3496','F3502','F3504','F3532','F3640',
'F3748','F3800','F3801','F3805','F3811',
'F3898','F3908','F3912','F3913','F61',
'F844','F850','F886','F949','F996'
]

# ------------------------------------------
# PREPARE DATA
# ------------------------------------------

X = df[selected_features].copy()

y = df[target_col]

# ------------------------------------------
# ENCODE CATEGORICAL FEATURES
# ------------------------------------------

for col in X.columns:

    if X[col].dtype == "object":

        X[col] = (
            X[col]
            .astype("category")
            .cat.codes
        )

print("\nFeature Matrix Shape:")
print(X.shape)

# ==========================================
# MUTUAL INFORMATION
# ==========================================

print("\n" + "=" * 60)
print("MUTUAL INFORMATION")
print("=" * 60)

mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi_df = pd.DataFrame({

    "Feature": X.columns,
    "MI_Score": mi_scores

})

mi_df = mi_df.sort_values(
    by="MI_Score",
    ascending=False
)

print("\nTOP 20 MI FEATURES")

print(mi_df.head(20))

# ==========================================
# RANDOM FOREST IMPORTANCE
# ==========================================

print("\n" + "=" * 60)
print("RANDOM FOREST IMPORTANCE")
print("=" * 60)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

rf_df = pd.DataFrame({

    "Feature": X.columns,
    "RF_Importance": rf.feature_importances_

})

rf_df = rf_df.sort_values(
    by="RF_Importance",
    ascending=False
)

print("\nTOP 20 RF FEATURES")

print(rf_df.head(20))

# ==========================================
# COMBINE RANKINGS
# ==========================================

mi_top = set(
    mi_df.head(30)["Feature"]
)

rf_top = set(
    rf_df.head(30)["Feature"]
)

final_features = sorted(
    list(
        mi_top.union(rf_top)
    )
)

print("\n" + "=" * 60)
print("FINAL FEATURE SELECTION")
print("=" * 60)

print("Total Final Features:",
      len(final_features))

for feature in final_features:
    print(feature)

# ==========================================
# SAVE FINAL FEATURES
# ==========================================

pd.DataFrame({

    "Selected_Features":
    final_features

}).to_csv(

    "../reports/final_selected_features.csv",
    index=False

)

print("\nFinal Feature List Saved")

# ==========================================
# SUMMARY
# ==========================================

print("\n" + "=" * 60)
print("FEATURE SELECTION COMPLETE")
print("=" * 60)

print("Original Features :",
      len(selected_features))

print("Final Features :",
      len(final_features))

print("=" * 60)

In [ ]:
pd.DataFrame({
    "Selected_Features": final_features
}).to_csv(
    "../data/processed/final_selected_features.csv",
    index=False
)

# Notebook 04 – Feature Selection

The objective of this notebook was to identify the most important features for mule account detection from the 95 candidate features obtained in Notebook 03.

Although 95 features were discovered during the feature exploration phase, not all of them contribute equally to identifying suspicious accounts. Using all features may increase model complexity, training time, and noise. Therefore, feature selection was performed to retain only the most informative features.

Two feature selection techniques were used:

### 1. Mutual Information (MI)

Mutual Information measures how much information a feature provides about the target variable (F3924). A feature with a higher MI score contains more useful information for distinguishing between legitimate and mule accounts.

For example, if the values of a feature are significantly different for normal and suspicious accounts, that feature receives a higher Mutual Information score because it helps identify mule account behavior.

The Mutual Information analysis showed that features such as F2230 and F3912 contained the highest amount of information related to the target variable and were therefore considered highly important.

### 2. Random Forest Feature Importance (RF)

Random Forest is a machine learning algorithm that builds multiple decision trees to classify data. During training, the algorithm automatically determines which features are most useful for making classification decisions.

Features that are frequently used by the decision trees and contribute significantly to correct predictions receive higher importance scores.

Random Forest Feature Importance therefore provides another perspective on feature usefulness by measuring how valuable each feature is during the classification process.

### Combining Both Methods

Both Mutual Information and Random Forest Feature Importance were used because each method evaluates features differently.

* Mutual Information measures the information content of a feature.
* Random Forest measures the practical usefulness of a feature during classification.

Features that performed well in either method were retained and combined to form the final feature set.

### Results

The feature selection process reduced the feature space from:

95 Candidate Features → 43 Final Features

This means that 52 less informative features were removed, while 43 highly relevant features were retained for further analysis and model development.

### Importance of Feature Selection

Feature selection helps improve machine learning performance by:

* Reducing noise in the dataset.
* Decreasing model training time.
* Lowering computational complexity.
* Improving model interpretability.
* Enhancing the ability to detect mule accounts accurately.

### Conclusion

Notebook 04 successfully identified the most informative features for mule account detection using Mutual Information and Random Forest Feature Importance. The process reduced the feature set from 95 candidate features to 43 high-quality features, which will be used in the subsequent model training phase.


In [5]:
import pandas as pd

# Load processed dataset
df = pd.read_csv("../data/processed/preprocessed_data.csv")

target_col = "F3924"

features = [
'F1057','F1165','F1381','F1387','F1393','F1489','F1495',
'F1501','F159','F1597','F1603','F1609','F162','F1707',
'F1713','F1717','F1719','F1755','F1814','F1815','F1819',
'F1821','F1825','F1827','F1861','F1863','F2230','F2486',
'F2489','F267','F2686','F270','F3484','F3532','F3640',
'F3748','F3800','F3801','F3805','F3811','F3898','F3912',
'F949'
]

print("=" * 60)
print("CHECKING FEATURES VS TARGET")
print("=" * 60)

for feature in features:

    print("\n" + "-" * 50)
    print("Feature:", feature)

    try:
        print(
            df.groupby(target_col)[feature]
            .agg(["mean", "median"])
        )

    except:
        print(
            df[feature].value_counts().head()
        )

CHECKING FEATURES VS TARGET

--------------------------------------------------
Feature: F1057
               mean    median
F3924                        
0      7.133979e+06  200000.0
1      1.009015e+05   38788.0

--------------------------------------------------
Feature: F1165
               mean    median
F3924                        
0      8.879305e+06  210000.0
1      1.237779e+05   40000.0

--------------------------------------------------
Feature: F1381
               mean    median
F3924                        
0      7.046987e+06  170000.0
1      1.008492e+05   38787.0

--------------------------------------------------
Feature: F1387
               mean    median
F3924                        
0      6.446004e+06  119247.0
1      9.672537e+04   36199.0

--------------------------------------------------
Feature: F1393
               mean    median
F3924                        
0      6.446004e+06  119247.0
1      9.672537e+04   36199.0

------------------------------------

In [7]:
import pandas as pd

df = pd.read_csv("../data/processed/preprocessed_data.csv")

for feature in [
    "F2230",
    "F3898",
    "F3912",
    "F3811",
    "F1821",
    "F1827"
]:
    print(feature, "->", df[feature].dtype)

F2230 -> object
F3898 -> int64
F3912 -> int64
F3811 -> float64
F1821 -> float64
F1827 -> float64


In [8]:
import pandas as pd

df = pd.read_csv("../data/processed/preprocessed_data.csv")

target_col = "F3924"

important_features = [
    "F2230",
    "F3898",
    "F3912",
    "F3811",
    "F1821",
    "F1827"
]

for feature in important_features:

    print("\n" + "="*60)
    print("FEATURE:", feature)
    print("="*60)

    print("Datatype:", df[feature].dtype)

    # Numerical
    if pd.api.types.is_numeric_dtype(df[feature]):

        print(
            df.groupby(target_col)[feature]
            .agg(["min","max","mean","median"])
        )

    # Categorical / Object
    else:

        print("\nTop Values:")

        print(
            pd.crosstab(
                df[feature],
                df[target_col]
            ).head(20)
        )


FEATURE: F2230
Datatype: object

Top Values:
F3924     0   1
F2230          
Dec25     0  10
Nov25     0  23
Oct25  9001   0
Sep25     0  48

FEATURE: F3898
Datatype: int64
       min  max      mean  median
F3924                            
0        0   57  1.867348     3.0
1        0    3  0.617284     1.0

FEATURE: F3912
Datatype: int64
       min  max      mean  median
F3924                            
0        0    1  0.000333     0.0
1        0    1  0.975309     1.0

FEATURE: F3811
Datatype: float64
           min           max          mean     median
F3924                                                
0         0.00  3.800514e+10  2.934421e+07  853831.48
1      1051.21  5.301864e+06  3.561288e+05  135589.98

FEATURE: F1821
Datatype: float64
           min           max          mean     median
F3924                                                
0         0.00  6.234266e+10  4.486484e+07  729742.86
1      2419.87  4.476267e+06  3.228267e+05  126234.51

FEATURE: F1827
Dataty

In [9]:
print(df["F2230"].dtype)
print(df["F2230"].nunique())
print(df["F2230"].value_counts().head(20))

object
4
F2230
Oct25    9001
Sep25      48
Nov25      23
Dec25      10
Name: count, dtype: int64
